In [1]:
# Installation des dépendances
!pip install -q \
    pandas \
    requests \
    rapidfuzz \
    openai \
    unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 13.3 MB/s eta 0:00:00


In [9]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("Clé OpenAI chargée.")

Clé OpenAI chargée.


In [11]:
import re, time, json, unicodedata
from functools import lru_cache
import requests, pandas as pd
try:
    from rapidfuzz import process, fuzz
    _HAS_RAPIDFUZZ = True
except ImportError:
    _HAS_RAPIDFUZZ = False


def normalize(text: str) -> str:
    """Minuscules + suppression accents + strip."""
    if not text:
        return ""
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return text


def normalize_upper(x) -> str:
    """Normalisation majuscule pour l'annuaire ANS """
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    x = unicodedata.normalize("NFD", x)
    x = "".join(c for c in x if unicodedata.category(c) != "Mn")
    x = re.sub(r"[^A-Z0-9 ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

In [12]:
def charger_referentiel_communes(timeout=60):
    r = requests.get("https://geo.api.gouv.fr/communes",
                     params={"fields": "nom", "format": "json"}, timeout=timeout)
    r.raise_for_status()
    return [c["nom"] for c in r.json()]

COMMUNES_REF = charger_referentiel_communes()
print("Communes chargées :", len(COMMUNES_REF))

SEUIL_AUTO, SEUIL_VERIF = 0.80, 0.45
COMM_NORM2OFF = {normalize(c): c for c in COMMUNES_REF}

def ville_propre(v):
    """Étape 'City status' : la valeur brute est-elle déjà une commune officielle ?"""
    if not isinstance(v, str) or not v.strip():
        return None
    return COMM_NORM2OFF.get(normalize(v))

def valider_deterministe(ville):
    """Validation déterministe : la ville proposée appartient-elle au référentiel ?"""
    if not ville:
        return None
    return COMM_NORM2OFF.get(normalize(ville))

def statut_final(ville_valide, confiance):
    if ville_valide is None:
        return "non_resolu"
    c = confiance or 0.0
    if c >= SEUIL_AUTO:
        return "corrige_auto"
    if c >= SEUIL_VERIF:
        return "a_verifier"
    return "non_resolu"

Communes chargées : 34969


In [13]:
def charger_annuaire_txt(path, sep="|",
                         col_nom="Nom d'exercice",
                         col_prenom="Prénom d'exercice",
                         col_ville="Libellé commune (coord. structure)",
                         col_profession="Libellé profession",
                         col_cp="Code postal (coord. structure)",
                         professions=None,
                         departements=None,
                         encoding="utf-8"):
    besoins = {col_nom, col_prenom, col_ville, col_profession, col_cp}
    df = pd.read_csv(path, sep=sep, dtype=str, encoding=encoding,
                     keep_default_na=False, on_bad_lines="skip",
                     usecols=lambda c: c in besoins)
    if professions and col_profession in df.columns:
        df = df[df[col_profession].isin(professions)]
    if departements and col_cp in df.columns:
        df = df[df[col_cp].str[:2].isin(departements)]
    if col_prenom in df.columns:
        nom = (df[col_nom].fillna("") + " " + df[col_prenom].fillna("")).str.strip()
    else:
        nom = df[col_nom].fillna("").str.strip()
    out = pd.DataFrame({"nom": nom, "ville": df[col_ville].fillna("").str.strip()})
    return out[out["ville"] != ""].drop_duplicates().reset_index(drop=True)

**system prompt**



In [14]:
SYSTEM_PROMPT = """Tu es un agent de correction du champ 'Ville' de comptes rendus RCP (cancérologie).
La valeur brute peut être : un nom de ville (souvent mal orthographié), un code postal
(5 ou 4 chiffres), une adresse postale, ou vide.

Ta mission : retrouver la commune française réelle, puis appeler soumettre_resultat.
- nom de ville -> rechercher_commune_par_nom
- code postal -> commune_depuis_code_postal (complète un code à 4 chiffres par un zéro initial)
- adresse -> ville_depuis_adresse
- VIDE -> lis le texte : trouves-y la ville de RÉSIDENCE du patient (pas les villes
  d'examens/hôpitaux) et valide-la via rechercher_commune_par_nom ; Sinon, repère le professionnel de santé référent
(médecin, infirmier, kinésithérapeute,pharmacien, chirurgien-dentiste,sage-femme, etc.)
et appelle rechercher_professionnel_annuaire pour retrouver sa commune d'exercice.
IGNORE les professionnels anonymisés ("Dr X", "Docteur X", "Mme X") : ne les
cherche JAMAIS dans l'annuaire ; si c'est le seul professionnel mentionné,
soumets ville=null.

Choisis toi-même les outils. Quand tu as la réponse, appelle soumettre_resultat avec :
- ville : la commune officielle (ou null si rien n'aboutit)
- methode ∈ {referentiel_nom, fuzzy_local, code_postal, ban_adresse, texte_hdlm,
  annuaire_medecin, non_resolu}
- type_degradation ∈ {nom_errone, code_postal, adresse, vide_texte, vide_medecin, inconnu}
- entite : l'information brute exploitée (nom mal écrit, code postal, adresse,
  ville trouvée dans le texte, ou nom du médecin référent)
- confiance entre 0 et 1.
Si rien n'aboutit : methode='non_resolu', ville=null, type_degradation='inconnu',
entite=null, confiance=0.

IMPORTANT : tu disposes d'un nombre limité d'actions. Au plus tard à ta 4e action,
tu DOIS appeler soumettre_resultat avec le meilleur résultat obtenu jusque-là
(ou ville=null si aucune piste n'a abouti). Ne répète jamais un appel d'outil
qui a déjà échoué avec les mêmes arguments."""


class AgentVilleToolUsing:
    API_GEO = "https://geo.api.gouv.fr/communes"
    API_BAN = "https://api-adresse.data.gouv.fr/search"

    def __init__(self, client, annuaire_df=None, communes_ref=None,
                 model="gpt-4o-mini", max_iter=8,
                 seuil_medecin=80, seuil_commune_local=80,
                 timeout=10, pause=0.2, retries=3):
        self.client, self.model, self.max_iter = client, model, max_iter
        self.seuil_medecin = seuil_medecin
        self.seuil_commune_local = seuil_commune_local
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": "CANEXPOVAL-agent-tooluse/1.0"})
        self.timeout, self.pause, self.retries = timeout, pause, retries
        if annuaire_df is not None:
            self.annuaire = annuaire_df.copy()
            self.annuaire["_nom_norm"] = self.annuaire["nom"].map(
                lambda s: normalize_upper(s).lower()
            )
            self._noms_norm = self.annuaire["_nom_norm"].tolist()
            self._index_exact = dict(zip(self.annuaire["_nom_norm"], self.annuaire["ville"]))
        else:
            self.annuaire, self._noms_norm, self._index_exact = None, [], {}
        if communes_ref is not None:
            self._comm_officiels = list(communes_ref)
            self._comm_norm = [normalize(c) for c in communes_ref]
        else:
            self._comm_officiels, self._comm_norm = [], []

    def _get(self, url, params):
        for k in range(self.retries):
            try:
                r = self.session.get(url, params=params, timeout=self.timeout)
                if r.status_code == 429:
                    time.sleep(self.pause * (2 ** k) + 0.5); continue
                r.raise_for_status(); time.sleep(self.pause)
                return r.json()
            except (requests.RequestException, ValueError):
                time.sleep(self.pause * (2 ** k))
        return None

    @lru_cache(maxsize=8192)
    def _t_nom(self, nom):
        data = self._get(self.API_GEO, {"nom": nom, "fields": "nom,population",
                                        "boost": "population", "limit": 5})
        if data:
            if _HAS_RAPIDFUZZ:
                best = max(data, key=lambda c: fuzz.token_sort_ratio(normalize(nom), normalize(c["nom"])))
                sc = fuzz.token_sort_ratio(normalize(nom), normalize(best["nom"]))
                return {"commune": best["nom"], "source": "geo.api", "score": sc}
            return {"commune": data[0]["nom"], "source": "geo.api", "score": None}
        if _HAS_RAPIDFUZZ and self._comm_norm:
            m = process.extractOne(normalize(nom), self._comm_norm,
                                   scorer=fuzz.token_sort_ratio, score_cutoff=self.seuil_commune_local)
            if m:
                return {"commune": self._comm_officiels[m[2]], "source": "fuzzy_local", "score": m[1]}
        return {"commune": None, "source": None, "score": None}

    @lru_cache(maxsize=8192)
    def _t_cp(self, cp):
        data = self._get(self.API_GEO, {"codePostal": str(cp).zfill(5), "fields": "nom,population"})
        if not data:
            return {"commune": None}
        return {"commune": max(data, key=lambda c: c.get("population", 0) or 0)["nom"]}

    @lru_cache(maxsize=8192)
    def _t_adresse(self, adresse):
        data = self._get(self.API_BAN, {"q": adresse, "limit": 1})
        feats = (data or {}).get("features", [])
        if not feats:
            return {"commune": None}
        p = feats[0]["properties"]
        return {"commune": p.get("city"), "score": p.get("score")}

    def _t_professionnel(self, nom):
        if self.annuaire is None or not nom:
            return {"commune": None}
        cible = normalize_upper(nom).lower()
        # Rejet des formes anonymisées ("dr x") ou trop courtes
        mots_utiles = [m for m in cible.split()
                       if m not in {"dr", "docteur", "dre", "mme", "m",
                                    "madame", "monsieur", "pr", "professeur"}]
        if not mots_utiles or all(len(m) <= 1 for m in mots_utiles):
            return {"commune": None, "match": "rejete_anonyme"}
        cible = " ".join(mots_utiles)
        if cible in self._index_exact:
            return {"commune": self._index_exact[cible], "match": "exact"}
        if _HAS_RAPIDFUZZ and self._noms_norm:
            m = process.extractOne(cible, self._noms_norm, scorer=fuzz.token_sort_ratio,
                                   score_cutoff=self.seuil_medecin)
            if m:
                return {"commune": self._index_exact[m[0]], "match": "fuzzy", "score": m[1]}
        return {"commune": None}

    def _tools_schema(self):
        obj = lambda **p: {"type": "object", "properties": p, "required": list(p)}
        s = lambda d: {"type": "string", "description": d}
        return [
            {"type": "function", "function": {"name": "rechercher_commune_par_nom",
                "description": "Trouve la commune réelle à partir d'un nom (même mal orthographié).",
                "parameters": obj(nom=s("nom de ville tel qu'écrit"))}},
            {"type": "function", "function": {"name": "commune_depuis_code_postal",
                "description": "Renvoie la commune d'un code postal (5 chiffres).",
                "parameters": obj(code_postal=s("code postal à 5 chiffres"))}},
            {"type": "function", "function": {"name": "ville_depuis_adresse",
                "description": "Géocode une adresse et renvoie la ville la plus probable.",
                "parameters": obj(adresse=s("adresse libre"))}},
            {"type": "function", "function": {"name": "rechercher_professionnel_annuaire",
    "description": "Cherche un professionnel de santé dans l'annuaire et renvoie sa ville d'exercice.",
    "parameters": obj(nom_professionnel=s("nom et prénom du professionnel de santé"))}},
            {"type": "function", "function": {"name": "soumettre_resultat",
                "description": "Soumet le résultat final structuré et termine la tâche.",
                "parameters": {"type": "object", "properties": {
                    "ville": {"type": ["string", "null"]},
                    "methode": {"type": "string"},
                    "type_degradation": {"type": "string"},
                    "entite": {"type": ["string", "null"]},
                    "confiance": {"type": "number"}},
                    "required": ["ville", "methode", "type_degradation",
                                 "entite", "confiance"]}}},
        ]

    def _execute(self, name, args):
        if name == "rechercher_commune_par_nom":
            return self._t_nom(args.get("nom", ""))
        if name == "commune_depuis_code_postal":
            return self._t_cp(args.get("code_postal", ""))
        if name == "ville_depuis_adresse":
            return self._t_adresse(args.get("adresse", ""))
        if name == "rechercher_professionnel_annuaire":
            return self._t_professionnel(args.get("nom_professionnel", ""))
        return {"erreur": "outil inconnu"}

    def corriger(self, ville_brute, texte_hdlm=""):
        user = (f"Valeur brute du champ Ville : {ville_brute!r}\n\n"
                f"Texte (Histoire de la maladie), utile si le champ est vide :\n"
                f"{str(texte_hdlm)[:6000]}")
        messages = [{"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user}]
        tools = self._tools_schema()
        t0, n_calls = time.time(), 0
        for _ in range(self.max_iter):
            resp = self.client.chat.completions.create(
                model=self.model, messages=messages, tools=tools,
                tool_choice="required", temperature=0)
            n_calls += 1
            msg = resp.choices[0].message
            messages.append(msg)
            if not msg.tool_calls:
                continue
            for tc in msg.tool_calls:
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except json.JSONDecodeError:
                    args = {}
                if tc.function.name == "soumettre_resultat":
                    return {"ville_brute": ville_brute,
                            "ville_corrigee": args.get("ville"),
                            "methode": args.get("methode", "non_resolu"),
                            "type_degradation": args.get("type_degradation", "inconnu"),
                            "entite": args.get("entite"),
                            "confiance": args.get("confiance"),
                            "n_appels_llm": n_calls,
                            "latence_s": round(time.time() - t0, 2)}
                result = self._execute(tc.function.name, args)
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(result, ensure_ascii=False)})
        return {"ville_brute": ville_brute, "ville_corrigee": None,
                "methode": "non_resolu", "type_degradation": "inconnu",
                "entite": None, "confiance": 0.0,
                "n_appels_llm": n_calls, "latence_s": round(time.time() - t0, 2)}


# FONCTION MODULE
def corriger_pipeline(agent, df, col_ville="Ville", col_texte="Histoire de la maladie"):
    lignes = []
    for _, row in df.iterrows():
        brut = row.get(col_ville, "")
        officielle = ville_propre(brut)
        if officielle:
            lignes.append({"ville_corrigee": officielle, "methode": "deja_propre",
                           "type_degradation": "aucune", "entite": brut,
                           "confiance": 1.0, "valide": True, "statut": "saine",
                           "n_appels_llm": 0, "latence_s": 0.0})
            continue
        res = agent.corriger(brut, row.get(col_texte, ""))
        validee = valider_deterministe(res["ville_corrigee"])
        res["ville_corrigee"] = validee or res["ville_corrigee"]
        res["valide"] = validee is not None
        res["statut"] = statut_final(validee, res.get("confiance"))
        lignes.append(res)
    cols = ["ville_corrigee", "methode", "type_degradation", "entite",
            "confiance", "valide", "statut", "n_appels_llm", "latence_s"]
    return df.join(pd.DataFrame(lignes, index=df.index)[cols])

In [15]:
from openai import OpenAI
client = OpenAI()

df = pd.read_csv("rcp_data_degrade_v2.csv", sep=";", dtype=str, keep_default_na=False)
print("RCP :", df.shape)

annuaire = charger_annuaire_txt(
    "annuaire.txt", sep="|",
    col_nom="Nom d'exercice",
    col_prenom="Prénom d'exercice",
    col_ville="Libellé commune (coord. structure)",
    departements=None,
)
print("Annuaire :", annuaire.shape)

RCP : (100, 17)
Annuaire : (1573313, 2)


In [16]:
# Création de l'agent
agent = AgentVilleToolUsing(
    client=client,
    annuaire_df=annuaire,
    communes_ref=COMMUNES_REF
)

# Correction du DataFrame
resultats = corriger_pipeline(agent, df)

# Affichage des résultats
resultats[
    [
        "Ville",
        "ville_corrigee",
        "methode",
        "confiance",
        "n_appels_llm",
        "latence_s",
    ]
].head(20)

,Ville,ville_corrigee,methode,confiance,n_appels_llm,latence_s
0,Charritte-de-Bas,Charritte-de-Bas,deja_propre,1.000000,0,0.00
1,,Claye-Souilly,annuaire_medecin,0.857143,2,5.97
2,Roulles,Roullens,fuzzy_local,0.933333,2,2.95
3,,Tours,texte_hdlm,1.000000,2,4.64
4,9460,Quérigut,code_postal,1.000000,2,2.21
5,,None,non_resolu,0.000000,2,2.51
6,96 Avenue de l'Europe,Carpentras,ban_adresse,0.970000,2,4.27
7,75 Rue des Tanneurs,Lys-lez-Lannoy,ban_adresse,0.966755,2,2.55
8,28200,None,non_resolu,0.000000,4,8.67
9,,None,non_resolu,0.000000,2,4.72


**Evaluation**

In [17]:
import pandas as pd
import numpy as np
import re
import unicodedata


# NORMALIZATION

def normalize_eval(value):
    """
    Normalize city names before comparison:
    - lowercase
    - remove accents
    - remove unnecessary punctuation
    - normalize spaces
    """
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )

    text = re.sub(r"[^a-z0-9\s\-']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def _fix_mojibake(value):
    """
    Attempt to correct encoding problems such as:
    vÃ©ritÃ© -> vérité
    """
    if not isinstance(value, str):
        return value

    try:
        return value.encode("latin1").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return value


def _is_missing_eval(value):
    """Return True when a predicted value is missing."""
    if value is None:
        return True

    if pd.isna(value):
        return True

    return str(value).strip().lower() in {
        "",
        "nan",
        "none",
        "null"
    }


def _safe_mean(series):
    """Compute a mean while safely handling empty series."""
    if len(series) == 0:
        return np.nan

    return series.mean()


# MAIN EVALUATION FUNCTION

def evaluate_with_ground_truth(
    df_result,
    ground_truth_path="/content/rcp_data_degrade_v2_solution.csv",
    architecture="MAS",
    validator=None,
    export_path=None
):

    out = df_result.copy().reset_index(drop=True)

    # 1. CHECK REQUIRED COLUMNS

    required_columns = {
        "Ville",
        "ville_enrichie",
        "ville_statut"
    }

    missing_columns = required_columns.difference(out.columns)

    if missing_columns:
        raise ValueError(
            "Colonnes absentes dans df_result : "
            f"{sorted(missing_columns)}"
        )

    # 2. READ GROUND TRUTH

    try:
        gt = pd.read_csv(
            ground_truth_path,
            sep=";",
            dtype=str,
            keep_default_na=False,
            skip_blank_lines=False,
            encoding="utf-8-sig"
        ).reset_index(drop=True)

    except UnicodeDecodeError:
        gt = pd.read_csv(
            ground_truth_path,
            sep=";",
            dtype=str,
            keep_default_na=False,
            skip_blank_lines=False,
            encoding="latin1"
        ).reset_index(drop=True)

    gt.columns = [
        _fix_mojibake(column.strip())
        for column in gt.columns
    ]

    for column in gt.columns:
        gt[column] = gt[column].map(_fix_mojibake)

    if "Ville vérité" not in gt.columns:
        raise ValueError(
            "La colonne 'Ville vérité' est absente du fichier de vérité terrain. "
            f"Colonnes disponibles : {gt.columns.tolist()}"
        )

    if len(out) != len(gt):
        raise ValueError(
            "Les fichiers n'ont pas le même nombre de lignes : "
            f"résultats={len(out)}, vérité terrain={len(gt)}"
        )

    # 3. ALIGN RESULTS WITH GROUND TRUTH

    out["ville_reference"] = (
        gt["Ville vérité"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    annotated_mask = out["ville_reference"].ne("")

    print(
        f"Lignes annotées : "
        f"{int(annotated_mask.sum())} / {len(out)}"
    )

    out["_pred_norm"] = out["ville_enrichie"].map(normalize_eval)
    out["_ref_norm"] = out["ville_reference"].map(normalize_eval)
    out["_orig_norm"] = out["Ville"].map(normalize_eval)

    # True when the final prediction exactly matches the ground truth.
    out["ville_correcte"] = pd.Series(
        pd.NA,
        index=out.index,
        dtype="boolean"
    )

    out.loc[annotated_mask, "ville_correcte"] = (
        out.loc[annotated_mask, "_pred_norm"]
        ==
        out.loc[annotated_mask, "_ref_norm"]
    )

    # 4. DEFINE PIPELINE OUTCOMES

    resolved_statuses = {
        "saine",
        "corrige_auto",
        "a_verifier"
    }

    resolved_mask = (
        out["ville_statut"].isin(resolved_statuses)
        &
        ~out["ville_enrichie"].map(_is_missing_eval)
    )

    # Only actual automatic corrections.
    # "saine" is not counted as a correction.
    auto_correction_mask = (
        out["ville_statut"].eq("corrige_auto")
        &
        ~out["ville_enrichie"].map(_is_missing_eval)
    )

    review_mask = out["ville_statut"].eq("a_verifier")
    unresolved_mask = out["ville_statut"].eq("non_resolu")

    out["resolved"] = resolved_mask
    out["automatic_correction"] = auto_correction_mask

    # 5. REFERENTIAL VALIDATION

    if validator is None:
        raise ValueError(
            "Vous devez fournir une fonction validator. Exemple :\n"
            "validator=lambda city: referential.match_city(city) is not None"
        )

    out["valide_referentiel"] = False

    predictions_mask = ~out["ville_enrichie"].map(_is_missing_eval)

    out.loc[predictions_mask, "valide_referentiel"] = (
        out.loc[predictions_mask, "ville_enrichie"]
        .apply(lambda city: bool(validator(city)))
    )

    # 6. EVALUATION SUBSETS

    evaluated = out.loc[annotated_mask].copy()

    evaluated_resolved = evaluated.loc[
        evaluated["resolved"]
    ].copy()

    evaluated_auto = evaluated.loc[
        evaluated["automatic_correction"]
    ].copy()

    evaluated_predictions = evaluated.loc[
        ~evaluated["ville_enrichie"].map(_is_missing_eval)
    ].copy()

    # 7. MAIN METRICS
    # 7.1 Resolution Rate

    resolution_rate = (
        evaluated["resolved"].mean()
        if len(evaluated) > 0
        else np.nan
    )

    # 7.2 End-to-End Accuracy
        end_to_end_accuracy = (
        evaluated["ville_correcte"]
        .fillna(False)
        .astype(bool)
        .mean()
        if len(evaluated) > 0
        else np.nan
    )

    # 7.3 Accuracy on Resolved Records

    accuracy_resolved = (
        evaluated_resolved["ville_correcte"]
        .fillna(False)
        .astype(bool)
        .mean()
        if len(evaluated_resolved) > 0
        else np.nan
    )

    # 7.4 Referential Validation Rate

    referential_validation_rate = (
        evaluated_predictions["valide_referentiel"].mean()
        if len(evaluated_predictions) > 0
        else np.nan
    )

    # 7.5 Auto-Correction Accuracy

    auto_correction_accuracy = (
        evaluated_auto["ville_correcte"]
        .fillna(False)
        .astype(bool)
        .mean()
        if len(evaluated_auto) > 0
        else np.nan
    )

    # 8. EFFICIENCY METRICS

    if "n_appels_llm" in out.columns:
        llm_calls = pd.to_numeric(
            out["n_appels_llm"],
            errors="coerce"
        )

        total_llm_calls = llm_calls.sum(min_count=1)
        mean_llm_calls = llm_calls.mean()

    else:
        total_llm_calls = np.nan
        mean_llm_calls = np.nan

    if "latence_s" in out.columns:
        latency = pd.to_numeric(
            out["latence_s"],
            errors="coerce"
        )

        mean_latency = latency.mean()
        median_latency = latency.median()
        p95_latency = latency.quantile(0.95)

    else:
        mean_latency = np.nan
        median_latency = np.nan
        p95_latency = np.nan

    # 9. OPTIONAL DESCRIPTIVE RATES

    auto_correction_rate = (
        evaluated["automatic_correction"].mean()
        if len(evaluated) > 0
        else np.nan
    )

    human_verification_rate = (
        review_mask.loc[evaluated.index].mean()
        if len(evaluated) > 0
        else np.nan
    )

    unresolved_rate = (
        unresolved_mask.loc[evaluated.index].mean()
        if len(evaluated) > 0
        else np.nan
    )

    # 10. SUMMARY TABLE

    summary = pd.DataFrame([{
        "Architecture": architecture,

        "N Total": len(out),
        "N Annotated": len(evaluated),
        "N Resolved": len(evaluated_resolved),
        "N Automatic Corrections": len(evaluated_auto),

        "Resolution Rate": resolution_rate,
        "End-to-End Accuracy": end_to_end_accuracy,
        "Accuracy on Resolved Records": accuracy_resolved,
        "Referential Validation Rate": referential_validation_rate,
        "Auto-Correction Accuracy": auto_correction_accuracy,

        "Auto-Correction Rate": auto_correction_rate,
        "Human Verification Rate": human_verification_rate,
        "Unresolved Rate": unresolved_rate,

        "Total LLM Calls": total_llm_calls,
        "Mean LLM Calls per Record": mean_llm_calls,

        "Mean Latency (s)": mean_latency,
        "Median Latency (s)": median_latency,
        "P95 Latency (s)": p95_latency
    }])

    # 11. PERFORMANCE BY DEGRADATION TYPE

    if "ville_type_degradation" in evaluated.columns:

        robustness_rows = []

        for degradation_type, group in evaluated.groupby(
            "ville_type_degradation",
            dropna=False
        ):
            group_resolved = group.loc[group["resolved"]]
            group_auto = group.loc[group["automatic_correction"]]
            group_predictions = group.loc[
                ~group["ville_enrichie"].map(_is_missing_eval)
            ]

            robustness_rows.append({
                "Degradation Type": degradation_type,
                "N": len(group),

                "Resolution Rate": group["resolved"].mean(),

                "End-to-End Accuracy": (
                    group["ville_correcte"]
                    .fillna(False)
                    .astype(bool)
                    .mean()
                ),

                "Accuracy on Resolved Records": (
                    group_resolved["ville_correcte"]
                    .fillna(False)
                    .astype(bool)
                    .mean()
                    if len(group_resolved) > 0
                    else np.nan
                ),

                "Referential Validation Rate": (
                    group_predictions["valide_referentiel"].mean()
                    if len(group_predictions) > 0
                    else np.nan
                ),

                "Auto-Correction Accuracy": (
                    group_auto["ville_correcte"]
                    .fillna(False)
                    .astype(bool)
                    .mean()
                    if len(group_auto) > 0
                    else np.nan
                )
            })

        robustness = pd.DataFrame(robustness_rows)

    else:
        robustness = pd.DataFrame()

    # 12. ERROR ANALYSIS

    error_columns = [
        column
        for column in [
            "Ville",
            "ville_enrichie",
            "ville_reference",
            "ville_statut",
            "ville_score",
            "ville_type_degradation",
            "ville_source",
            "ville_decision_log",
            "methode",
            "entite",
            "valide_referentiel",
            "n_appels_llm",
            "latence_s"
        ]
        if column in evaluated.columns
    ]

    errors = evaluated.loc[
        ~evaluated["ville_correcte"]
        .fillna(False)
        .astype(bool),
        error_columns
    ].copy()

    # 13. DISPLAY RESULTS

    print("\nRÉSULTATS GLOBAUX")
    display(summary.T)

    if not robustness.empty:
        print("\nPERFORMANCE PAR TYPE DE DÉGRADATION")
        display(robustness)

    print(
        f"\nErreurs : {len(errors)} / {len(evaluated)}"
    )

    # 14. EXPORT

    if export_path:
        with pd.ExcelWriter(
            export_path,
            engine="openpyxl"
        ) as writer:

            summary.to_excel(
                writer,
                sheet_name="Summary",
                index=False
            )

            if not robustness.empty:
                robustness.to_excel(
                    writer,
                    sheet_name="By_Degradation_Type",
                    index=False
                )

            errors.to_excel(
                writer,
                sheet_name="Errors",
                index=False
            )

            out.to_excel(
                writer,
                sheet_name="Detailed_Results",
                index=False
            )

        print(f"Résultats exportés vers : {export_path}")

    return {
        "summary": summary,
        "robustness": robustness,
        "errors": errors,
        "detailed_results": out
    }

In [22]:

resultats_eval = resultats.rename(columns={
    "ville_corrigee": "ville_enrichie",
    "statut": "ville_statut",
    "confiance": "ville_score",
    "type_degradation": "ville_type_degradation",
    "methode": "ville_source"
}).copy()

evaluation_mono = evaluate_with_ground_truth(
    df_result=resultats_eval,
    ground_truth_path="/content/rcp_data_degrade_v2_solution.csv",
    architecture="Monolithic",
    validator=lambda city: valider_deterministe(city) is not None,
    export_path="/content/evaluation_monolithic.xlsx"
)

Lignes annotées : 97 / 100

RÉSULTATS GLOBAUX


,0
Architecture,Monolithic
N Total,100
N Annotated,97
N Resolved,84
N Automatic Corrections,77
Resolution Rate,0.865979
End-to-End Accuracy,0.597938
Accuracy on Resolved Records,0.690476
Referential Validation Rate,1.0
Auto-Correction Accuracy,0.662338



PERFORMANCE PAR TYPE DE DÉGRADATION


,Degradation Type,N,Resolution Rate,End-to-End Accuracy,Accuracy on Resolved Records,Referential Validation Rate,Auto-Correction Accuracy
0,adresse,15,1.0,0.200000,0.200000,1.0,0.200000
1,aucune,7,1.0,1.000000,1.000000,1.0,NaN
2,code_postal,21,1.0,0.619048,0.619048,1.0,0.619048
3,inconnu,13,0.0,0.000000,NaN,NaN,NaN
4,nom_errone,17,1.0,0.941176,0.941176,1.0,0.941176
5,vide_medecin,14,1.0,0.714286,0.714286,1.0,0.714286
6,vide_texte,10,1.0,0.900000,0.900000,1.0,0.900000



Erreurs : 39 / 97
Résultats exportés vers : /content/evaluation_monolithic.xlsx
